In [0]:
print("=" * 80)
print("GOLD LAYER - BUSINESS METRICS & AGGREGATIONS")
print("=" * 80)

# ============ READ SILVER TABLES ============
print("\nReading Silver tables")
df_silver_csv = spark.table("dm2_credit_risk.silver_loan_apps")
df_silver_json = spark.table("dm2_credit_risk.silver_world_bank")

print(f"silver_loan_apps: {df_silver_csv.count():,} rows")
print(f"silver_world_bank: {df_silver_json.count():,} rows")

# ============ CREATE GOLD LAYER WITH SQL ============
print("\n" + "=" * 80)
print("CREATING GOLD LAYER TABLES")
print("=" * 80)



In [0]:
%sql

-- Create Gold table: Risk Metrics by Segment
CREATE OR REPLACE TABLE dm2_credit_risk.gold_credit_risk_metrics AS
SELECT 
    CASE 
        WHEN credit_to_income_ratio > 3 THEN 'High Risk'
        WHEN credit_to_income_ratio > 1.5 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END as risk_segment,
    family_status,
    education_type,
    COUNT(SK_ID_CURR) as total_customers,
    SUM(CASE WHEN TARGET = 1 THEN 1 ELSE 0 END) as defaulted_customers,
    ROUND(SUM(CASE WHEN TARGET = 1 THEN 1 ELSE 0 END) / COUNT(SK_ID_CURR) * 100, 2) as default_rate_percent,
    ROUND(AVG(AMT_CREDIT), 2) as avg_credit_amount,
    ROUND(AVG(AMT_INCOME_TOTAL), 2) as avg_income,
    ROUND(AVG(credit_to_income_ratio), 2) as avg_credit_to_income_ratio,
    COUNT(DISTINCT SK_ID_CURR) as unique_customers
FROM dm2_credit_risk.silver_loan_apps
GROUP BY risk_segment, family_status, education_type
ORDER BY default_rate_percent DESC;

-- Verify Gold table
SELECT * FROM dm2_credit_risk.gold_credit_risk_metrics LIMIT 10;